In [20]:
import numpy as np
import pandas as pd
import paretoset
import time
import json
import heapq
import cvxpy as cp
# glpk cvxopt

PATH = '/Users/sasha/Downloads/my_jupyter/tutorial/cart/'
MAX_NUM_SEQ_ALGO = 50
CURR_N_BOXES = 11
MAX_N_BOXES = int(np.ceil(CURR_N_BOXES * 1.2))
DIM_FACTOR = 139
AVG_FILL_RATE = 0.6

MAX_BATCH_SIZE = 10

list2str = lambda l: ','.join([str(int(j)) for j in l])
str2list = lambda s: [bool(int(g)) for g in s.split(',')]

def CardStack(df):
    dims = [df['l'].max(), df['w'].max(), df['h'].sum()]
    dims.sort(reverse = True)
    return pd.DataFrame([dims], columns = ['l', 'w', 'h'])

def TwoItem(df):
    a = list(df.iloc[0])
    b = list(df.iloc[1])
    dims = []
    for i in range(3):
        for j in range(3):
            ac = a.copy()
            ae = ac.pop(i)
            bc = b.copy()
            be = bc.pop(j)
            dim = [ae + be, max(ac[0], bc[0]), max(ac[1], bc[1])]
            dim.sort(reverse = True)
            dims.append(dim)
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h'])
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def TwoItemSeq(df_in, k_max = 9):
    df = df_in.copy()
    df['v'] = df['l'] * df['w'] * df['h']
    df = df.sort_values('v')[['l', 'w', 'h']]
    dims = df.iloc[[0]]
    for r in range(1, len(df.index)):
        # try pairwise addition of the new item to current superitems
        row = df.iloc[[r]]
        dims = pd.concat([TwoItem(pd.concat([row, dims.loc[[rd]]])) for rd in dims.index])
        dims = dims[paretoset.paretoset(dims, ['min', 'min', 'min'])]
        # reduce the number of superitems
        dims['v'] = dims['l'] * dims['w'] * dims['h']
        dims = dims.sort_values('v')[['l', 'w', 'h']].iloc[0:k_max].reset_index(drop = True)
    return dims

def SingleSize(df, is_flat = False):
    if is_flat:
        n = df['q'].iloc[0]
    else:
        n = len(df.index)
    [l, w, h] = [df['l'].iloc[0], df['w'].iloc[0], df['h'].iloc[0]]
    rotations = [[l, w, h], [l, h, w], [w, l, h], [w, h, l], [h, l, w], [h, w, l]]
    dims = []
    for x in range(int(np.ceil(n ** (1/3)))):
        for y in range(x, int(np.ceil((n / (x + 1)) ** (1/2)))):
            z = int(np.ceil(n / (x + 1) / (y + 1)))
            if z >= y + 1:
                for r in rotations:
                    dim = [r[0] * (x + 1), r[1] * (y + 1), r[2] * z]
                    dim.sort(reverse = True)
                    dims.append(dim.copy())
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h'])
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def DimsToDict(dims):
    dims_dict = []
    for i in dims.index:
        for j in range(dims.loc[i, 'q']):
            dims_dict.append([dims.loc[i, 'p'], 
                              dims.loc[[i],['l', 'w', 'h']].copy(), 
                              dims.loc[i, 'l'] * dims.loc[i, 'w'] * dims.loc[i, 'h']])
    return {k : dims_dict[k] for k in range(len(dims_dict))}

def SizeGroups(df):
    odf = df.copy()
    odf['p'] = (odf['p'] * odf['q']).apply(lambda x: round(x, 2))
    odf = odf.groupby(['l', 'w', 'h'], as_index = False)[['q', 'p']].sum()
    dims_dict = {}
    for i in odf.index:
        dims_i = SingleSize(odf.loc[[i]], True)
        min_vol_i = (dims_i['l'] * dims_i['w'] * dims_i['h']).min()
        dims_dict[i] = [odf.loc[i, 'p'], dims_i, min_vol_i]
    return dims_dict

def EncompassingDims(dims1, dims2, extra = [], size = False):
    if size and 's' not in extra:
        extra.append('s')
    dims = pd.merge(dims1.assign(key = 1), dims2.assign(key = 1), on = 'key').drop('key', axis = 1)
    dims['l'] = dims[['l_x', 'l_y']].max(axis = 1)
    dims['w'] = dims[['w_x', 'w_y']].max(axis = 1)
    dims['h'] = dims[['h_x', 'h_y']].max(axis = 1)
    dims = dims[['l', 'w', 'h'] + extra]
    if not size:
        return dims[paretoset.paretoset(dims[['l', 'w', 'h']], ['min', 'min', 'min'])].reset_index(drop = True)
    else:
        return dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)

def FullyEncompassingDims(dims_dict):
    dims = pd.DataFrame([[0, 0, 0]], columns = ['l', 'w', 'h'])
    for key in dims_dict:
        dims = EncompassingDims(dims, dims_dict[key][1])
    return dims

def CrossTwoItem(dims1, dims2):
    dims = []
    columns = ['l', 'w', 'h']
    for i in dims1.index:
        for j in dims2.index:
            dims.append(TwoItem(pd.concat([dims1.loc[[i], columns], dims2.loc[[j], columns]], axis = 0)))
    dims = pd.concat(dims, axis = 0)
    return dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)

def MergeSmallest(dims_dict, max_d = None):
    sort_vol = pd.DataFrame([[key, dims_dict[key][2]] for key in dims_dict], columns = ['i', 'v']).sort_values('v')
    new_index = sort_vol['i'].max() + 1
    obj1 = dims_dict.pop(sort_vol.iloc[0,0])
    obj2 = dims_dict.pop(sort_vol.iloc[1,0])
    dims = CrossTwoItem(obj1[1], obj2[1])
    dims['v'] = dims[['l', 'w', 'h']].prod(axis = 1)
    min_vol = dims['v'].min()
    if max_d:
        dims = dims.sort_values('v').iloc[:max_d]
    dims_dict[new_index] = [obj1[0] + obj2[0], dims[['l', 'w', 'h']], min_vol]
    return dims_dict

def MergeSmallestSeq(dims_dict, max_d = None):
    all_dims = []
    max_shipments = len(dims_dict)
    for i in range(max_shipments):
        dims = FullyEncompassingDims(dims_dict)
        dims['s'] = len(dims_dict)
        dims['p'] = json.dumps([round(dims_dict[k][0], 2) for k in dims_dict])
        all_dims.append(dims)
        if len(dims_dict) > 1:
            dims_dict = MergeSmallest(dims_dict, max_d)
    all_dims = pd.concat(all_dims, axis = 0)
    return all_dims[paretoset.paretoset(all_dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)
    
def CheckFit(dims, boxes, many_to_1 = False):
    if many_to_1:
        group_cols = ['n', 'p']
    else:
        group_cols = ['n']
        
    fit = pd.merge(boxes.assign(key = 1), dims.assign(key = 1), on = 'key').drop('key', axis = 1)
    fit['f'] = (fit['L'] >= fit['l']) & (fit['W'] >= fit['w']) & (fit['H'] >= fit['h'])
    fit = fit[group_cols + ['f']].groupby(group_cols).max().reset_index()
    return fit

def MinMaxSum(lst, k):
    vals = np.sort(np.array(lst))[::-1]
    idxs = np.argsort(np.array(lst))[::-1]
    subsets = [(0, []) for _ in range(k)]
    heapq.heapify(subsets)
    for i in range(len(lst)):
        m = heapq.heappop(subsets)
        heapq.heappush(subsets, (m[0] + vals[i], m[1] + [idxs[i]]))
    return [(s, i) for s in range(k) for i in subsets[s][1]]

def CardStackMulti(df, max_box = None):
    l = df['l'].max()
    w = df['w'].max()
    hdf = pd.concat([pd.concat([df[df['q'] == q][['h', 'p']]] * q) for q in df['q'].drop_duplicates()]).reset_index(drop = True)
    dims = []
    if not max_box:
        max_box = df['q'].sum()
    for s in range(max_box):
        group_df = pd.DataFrame(MinMaxSum(list(hdf['h']), s + 1), columns = ['group', 'index']).set_index('index')
        hdf['group'] = group_df['group']
        h = hdf.groupby('group')['h'].sum().max()
        dim = [l, w, h]
        dim.sort(reverse = True)
        dims.append(dim + [s + 1, json.dumps(list(hdf.groupby('group')['p'].sum().apply(lambda x: round(x, 2))))])
    dims = pd.DataFrame(dims, columns = ['l', 'w', 'h', 's', 'p'])
    return dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)

def Flatten(df):
    odf = pd.concat([pd.concat([df[df['q'] == q]] * q) for q in df['q'].drop_duplicates()]).reset_index(drop = True)
    odf['q'] = 1
    return odf

def FullSearch(df, is_dict = False, multi_box = True):
    if not is_dict:
        n = len(df.index)
    else:
        n = len(df)
    masks = []
    lengths = []
    for i in range(1, 2 ** n):
        mask = [bool(int(b)) for b in list(str(bin(i))[2:])]
        mask = [False] * (n - len(mask)) + mask
        lengths.append(sum(mask))
        masks.append(mask)
    sort_masks = sorted(masks, key = sum)

    mask_dims = {}
    if multi_box:
        mult_dims = {}
    for m in sort_masks:
        k = sum(m)
        if k == 1:
            if not is_dict:
                item = df[pd.Series(m)][['l', 'w', 'h', 'p']].copy()
                mask_dims[list2str(m)] = [item[['l', 'w', 'h']], item['p'].sum()]
            else:
                item_id = sum([i * m[i] for i in range(n)])
                item = df[item_id][1].copy()
                item['p'] = df[item_id][0]
                mask_dims[list2str(m)] = [item[['l', 'w', 'h']], df[item_id][0]]
            if multi_box:
                item['s'] = 1
                item['p'] = item['p'].apply(lambda x: json.dumps([x]))
                mult_dims[list2str(m)] = item[['l', 'w', 'h', 's', 'p']]
        else:
            sub_masks = [sm[-k:] for sm in masks[:2 ** (k - 1) - 1]]
            active_pos = [i for i in range(n) if m[i]]
            dims = []
            mdims = []
            for sm in sub_masks:
                m1 = m.copy()
                m2 = m.copy()
                for pos in range(k):
                    m1[active_pos[pos]] = sm[pos]
                    m2[active_pos[pos]] = not sm[pos]
                [dims1, p1] = mask_dims[list2str(m1)]
                [dims2, p2] = mask_dims[list2str(m2)]
                dims.append(CrossTwoItem(dims1, dims2))
                if multi_box:
                    # multi-package dims are part 1 as a whole plus part 2 as partition
                    mdim2 = mult_dims[list2str(m2)]
                    mdim = EncompassingDims(dims1, mdim2, extra = ['p', 's'], size = True)
                    mdim['s'] = mdim['s'] + 1
                    mdim['p'] = mdim['p'].apply(lambda x: json.dumps([round(float(p1), 2)] + json.loads(x)))
                    mdims.append(mdim)
            dims = pd.concat(dims)
            dims = dims[paretoset.paretoset(dims, ['min', 'min', 'min'])].reset_index(drop = True)
            mask_dims[list2str(m)] = [dims, p1 + p2]
            if multi_box:
                # 2+package dims and 1-package dims
                mdim = dims.copy()
                mdim['s'] = 1
                mdim['p'] = json.dumps([round(float(p1 + p2), 2)])
                mdims.append(mdim)
                mdims = pd.concat(mdims)
                mdims = mdims[paretoset.paretoset(mdims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)
                mult_dims[list2str(m)] = mdims[['l', 'w', 'h', 's', 'p']]
    if multi_box:
        return mult_dims[list2str(sort_masks[-1])]
    out_dims = mask_dims[list2str(sort_masks[-1])][0]
    out_dims['s'] = 1
    out_dims['p'] = json.dumps([mask_dims[list2str(sort_masks[-1])][1]])
    return out_dims

def BoxMatch(actual, catalog, cols_match, act_id, tie_break_cols = [], p = 2):
    match = actual.merge(catalog, how = 'cross')
    match['diff'] = match.apply(lambda x: sum([abs(x[c_a] - x[cols_match[c_a]]) ** p for c_a in cols_match]) ** (1 / p), axis = 1)
    match = match.sort_values([act_id, 'diff'] + tie_break_cols).groupby(act_id).first().reset_index()
    return match



In [43]:
def InferTranCost(df):
    es = df.copy() 
    
    # if df['T'].notna().sum() > 1000...

    is_box = pd.concat([df[['n']], df[['L', 'W', 'H']].apply(lambda x: 
        'x'.join([str(c) for c in x]), axis = 1)], axis = 1).drop_duplicates().groupby('n').count()
    act_box_data = df[['n', 'L', 'W', 'H']].drop_duplicates().merge(is_box[is_box[0] == 1].reset_index()[['n']], on = 'n', how = 'right')
    match = BoxMatch(act_box_data, ul_bx, {'L_x' : 'L_y', 'W_x' : 'W_y', 'H_x' : 'H_y'}, 'n_x', ['c'], 2)
    es = es.merge(match[['n_x', 'c']], left_on = 'n', right_on = 'n_x', how = 'left')
    es['Box Cost'] = es['C'].combine_first(es['c'].fillna(0))

    es['W_s'] = es['P']
    es['W_i'] = es['p'] * es['q']
    es['V_i'] = es['l'] * es['w'] * es['h'] * es['q'] / AVG_FILL_RATE
    es['V_s'] = es['L'] * es['W'] * es['H']

    es['Shipment Cost'] = 0

    es_agg = es[['o', 'Shipment Cost', 'Box Cost', 'W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
    #es_agg = es_agg.groupby(['o', 'Shipment Cost', 'W_s', 'V_s'], as_index = False).sum()
    es_agg[['W_s', 'V_s', 'W_i', 'V_i']] = es_agg[['W_s', 'V_s', 'W_i', 'V_i']].fillna(0)
    es_agg['BW'] = es_agg.apply(lambda x: np.ceil(max(x['W_s'], x['W_i'],x['V_i']/DIM_FACTOR, (x['V_s'] if x['V_s'] else x['V_i']) / DIM_FACTOR)), axis = 1)

    es_agg['Shipment Cost'] = 6.0 + 0.3 * es_agg['BW']

    t = es_agg[es_agg['Shipment Cost'] > 0][['o', 'Shipment Cost', 'Box Cost', 'BW', 'V_i', 'V_s']]
    # display(t)

    y = 'Shipment Cost'
    x = 'BW'
    n = len(t.index)

    x_m = t[x].mean()
    y_m = t[y].mean()

    SLOPE = ((t[x] - x_m) * (t[y] - y_m)).sum() / ((t[x] - x_m) * (t[x] - x_m)).sum()
    INTERCEPT = y_m - SLOPE * x_m

    return [SLOPE, INTERCEPT, t, list(match['n_y'])]

def ComputeCost(odf, bx, o):
    num_items = odf['q'].sum()
    num_sizes = len(odf[['l', 'w', 'h']].drop_duplicates().index)
    
    dims = []
    dims.append(CardStackMulti(odf, 10))

    if num_items < 5:
        dims.append(FullSearch(Flatten(odf)))
    else:
        if num_items < 10:
            dims.append(MergeSmallestSeq(DimsToDict(odf), max_d = 10))
        if num_sizes < min(4, num_items):
            dims.append(FullSearch(SizeGroups(odf), is_dict = True))
        if 4 <= num_sizes < min(10, num_items): 
            dims.append(MergeSmallestSeq(SizeGroups(odf), max_d = 10))


    dims = pd.concat(dims, axis = 0)
    dims = dims[paretoset.paretoset(dims[['l', 'w', 'h', 's']], ['min', 'min', 'min', 'min'])].reset_index(drop = True)


    cost = CheckFit(dims, bx[['n', 'L', 'W', 'H']], many_to_1 = True).merge(bx, on = 'n')
    cost['V'] = cost['L'] * cost['W'] * cost['H']
    cost['o'] = o
    cost['t'] = cost.apply(lambda x: sum([FULFILLMENT_COST_P(x, p) for p in json.loads(x['p'])]), 
                           axis = 1) * cost['f'].apply(lambda x: 1 if x else None)
    cost['p'] = cost.apply(lambda x: x['p'] if x['f'] else None, axis = 1)
    cost = cost[['o', 'n', 't', 'p']].sort_values('t').groupby(['o', 'n'], as_index = False).first()
    return cost

def RunCartonization(in_df, ul_bx):
    df = in_df.copy()
    df = df.groupby(['o', 'l', 'w', 'h', 'p'], as_index = False).sum()
    df['key'] = df.apply(lambda x: '_'.join([str(round(x[c], 2)) for c in ['l', 'w', 'h', 'p', 'q']]), axis = 1)
    order_keys = df[['o', 'key']].sort_values('key').groupby('o', as_index = False).agg(lambda x: '+'.join(x))
    unique_orders = order_keys.groupby('key').first().reset_index(drop = True)

    df = in_df.merge(unique_orders, on = 'o', how = 'right')
    orders = df['o'].drop_duplicates()
    df = df.set_index('o')

    bx = ul_bx.copy()

    start = time.time()
    iterator = 0
    costs = []
    pounds = []
    print('unique orders', len(orders))
    for o in orders:
        odf = df.loc[[o]].reset_index(drop = True)

        num_items = odf['q'].sum()
        num_sizes = len(odf[['l', 'w', 'h']].drop_duplicates().index)

        na_flag = max(odf['l'].isna().sum(), odf['l'].isna().sum())
        if na_flag == 0:
            if num_items > MAX_BATCH_SIZE:
                batch_num = int(np.ceil(num_items / MAX_BATCH_SIZE))
                batch_size = int(np.ceil(num_items / batch_num))
                bodf = Flatten(odf).sort_values(['l', 'w', 'h'])
                cost_dfs = []
                for b in range(batch_num):
                    cost_dfs.append(ComputeCost(bodf.iloc[b * batch_size : (b + 1) * batch_size], bx, o))
                cost = pd.concat([cdf.set_index(['o', 'n']) for cdf in cost_dfs], axis = 1)
                cost['tt'] = cost[['t']].sum(axis = 1)
                cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
                cost['isna'] = cost[['t']].isna().any(axis = 1)
                cost = cost[['tt', 'pt', 'isna']].reset_index()
                cost['t'] = cost['tt'] * cost['isna'].apply(lambda x: 1 if not x else None)
                cost['p'] = cost.apply(lambda x: x['pt'] if not x['isna'] else None, axis = 1)
            else:
                cost = ComputeCost(odf, bx, o)

            costs.append(cost[['o', 'n', 't']].pivot(index = 'o', columns = 'n', values = 't'))
            pounds.append(cost[['o', 'n', 'p']].pivot(index = 'o', columns = 'n', values = 'p'))
            iterator += 1
            if iterator % 100 == 0:
                print(iterator, time.time() - start)
            #break

    print(time.time() - start)
    costs = pd.concat(costs, axis = 0)
    pounds = pd.concat(pounds, axis = 0)
    costs = costs.merge(order_keys, on = 'o', how = 'inner').drop(columns = ['o']).merge(order_keys, on = 'key', how = 'inner').drop(columns = 'key')
    pounds = pounds.merge(order_keys, on = 'o', how = 'inner').drop(columns = ['o']).merge(order_keys, on = 'key', how = 'inner').drop(columns = 'key')
    return [costs.set_index('o'), pounds.set_index('o')]

def BreakCosts(costs, pounds, ul_bx):
    box_costs = pounds.applymap(lambda x: len(json.loads(x)) if x else None)
    ul_bc = ul_bx[['n', 'c']].set_index('n')
    for col in box_costs:
        box_costs[col] *= ul_bc.loc[col, 'c']
    tran_costs = costs - box_costs
    return [box_costs, tran_costs]

def GreedySearch(ul_bx, costs, tran_costs):

    names = list(ul_bx['n'])

    current = [costs.count().sort_values(ascending = False).index[0]]
    min_cost = costs[current].min(axis = 1).sum()
    min_tran_cost = tran_costs[current].min(axis = 1).sum()

    output = [[1, round(min_cost, 2), round(min_tran_cost, 2), current[0]]]

    for i in range(1, MAX_N_BOXES):
        min_name = ''
        for n in names:
            c = costs[current + [n]].min(axis = 1).sum()
            if c < min_cost:
                min_cost = c
                min_tran_cost = tran_costs[current + [n]].min(axis = 1).sum()
                min_name = n
        if min_name != '':
            output.append([1 + i, round(min_cost, 2), round(min_tran_cost, 2), min_name])
            current += [min_name]
        else:
            break

    greedy_search = pd.DataFrame(output, columns = ['box_num', 'cumulative_cost', 'transport_cost', 'incremental_box'])
    greedy_search = greedy_search.merge(ul_bx[['n', 'L', 'W', 'H', 'c']], left_on = 'incremental_box', right_on = 'n', how = 'inner')
    greedy_search = greedy_search.drop(columns = ['n']).set_index('box_num')

    return [greedy_search, output]

def BreakevenPoint(df, costs, t, greedy_search):
    ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
    h_c = t.set_index('o').loc[ord_with_cost][['Shipment Cost']].sum().sum()
    h_b = t.set_index('o').loc[ord_with_cost][['Box Cost']].sum().sum()
    h_t = t.set_index('o').loc[ord_with_cost][['Shipment Cost', 'Box Cost']].sum().sum()

    for i in range(len(greedy_search.index)):
        parity_set = list(greedy_search['incremental_box'].iloc[:i + 1])
        c = tran_costs.loc[ord_with_cost][parity_set].min(axis = 1).sum()
        if c < h_c:
            break
    parity_count = i + 1
    return [parity_count, h_c, h_b, h_t]

def BoxAssortmentOptimization(costs, suite_size, is_integer = False, verbose = False):
    C = np.array(costs)
    [n, m] = [C.shape[0], C.shape[1]]
    if is_integer:
        r = cp.Variable(m, boolean = True)
    else:
        r = cp.Variable(m, nonneg = True)
    x = cp.Variable((n, m), nonneg = True)
    constraints = [x @ np.ones(m) == 1, x <= C, cp.sum(r) <= suite_size] + [x[:, i] <= r[i] for i in range(m)] 
    if not is_integer:
        constraints.append(r <= 1)

    prob = cp.Problem(cp.Minimize(cp.sum(cp.multiply(C, x))), constraints)
    
    if is_integer:
        prob.solve(solver = 'GLPK_MI', verbose = verbose, max_iters = 1000, abstol = 10 ** (-3), reltol = 10 ** (-3))
    else:
        prob.solve(solver = 'ECOS', verbose = verbose, max_iters = 1000, abstol = 10 ** (-3), reltol = 10 ** (-3))
    
    return [round(prob.value, 3), r.value]

def RunOptimization(costs, suite_sizes):

    eff_bx = list(costs.reset_index().melt(id_vars = ['o'], 
      var_name = ['n'], value_name = 'c_s').sort_values('c_s').groupby('o').first()['n'].drop_duplicates())

    sm = costs[eff_bx].fillna(0)
    sm = sm[sm.sum(axis = 1) > 0]
    sm['all'] = sm.agg(list, axis = 1).apply(json.dumps)
    sm = sm.groupby('all', as_index = False).sum().drop(columns = 'all')
    sm = sm[sm.sum(axis = 1) > 0]

    bao = {}

    start = time.time()
    for suite_size in suite_sizes:

        [obj, rv] = BoxAssortmentOptimization(sm, suite_size, is_integer = False)

        frac_ba = [eff_bx[i] for i in range(len(rv)) if rv[i] > 10 ** (-3)]
        if len(frac_ba) > suite_size:
            ism = sm[frac_ba]
            [iobj, irv] = BoxAssortmentOptimization(ism, suite_size, is_integer = True)
            iba = [frac_ba[i] for i in range(len(irv)) if irv[i] > 0.5]
        else:
            iobj = obj
            iba = frac_ba
        bao[suite_size] = (obj, iobj, iba)
        print(suite_size, time.time() - start)

    return bao

def Utilization(costs, pounds, boxes, bx):
    choices = costs[boxes].reset_index().melt(id_vars = ['o'], var_name = 'n', value_name = 't').sort_values('t').groupby('o', as_index = False).first()
    choices = choices.merge(pounds[boxes].reset_index().melt(id_vars = ['o'], var_name = 'n', value_name = 'p'), on = ['o', 'n'], how = 'left')
    choices['s'] = choices['p'].apply(lambda x: len(json.loads(x)) if x is not None else None)
    choices = choices.groupby('n', as_index = False).sum().sort_values('s', ascending = False)
    choices = choices.merge(bx[['n', 'L', 'W', 'H', 'c']], on = 'n', how = 'left')
    choices['Dims'] = choices.apply(lambda x: (str(x['L']) + 'x' + str(x['W']) + 'x' + str(x['H']) + 'x').replace('.0x', 'x')[:-1], axis = 1)
    choices['Cumulative Utilization Pct'] = (choices['s'].cumsum() / choices['s'].sum()).apply(lambda x: str(round(x * 100, 1)) + '%')
    return choices[['n', 's', 'Cumulative Utilization Pct', 'Dims', 't', 'L', 'W', 'H', 'c']].rename(columns = {'s' : 'Total Utilization', 'n' : 'Box Name'})


In [29]:
ul_bx = pd.read_csv(PATH + 'arka_boxes.csv')

df = pd.read_csv(PATH + 'data.csv')
df_baseline = pd.read_csv(PATH + 'data_baseline.csv')
es_df = df[['o', 'l', 'w', 'h', 'q', 'p']].copy()

[SLOPE, INTERCEPT, t, baseline_ba] = InferTranCost(df_baseline)

FULFILLMENT_COST = lambda x: x['c'] + INTERCEPT + SLOPE * np.ceil(max(x['p'] + x['P'], x['V'] / DIM_FACTOR))
FULFILLMENT_COST_P = lambda x, p: x['c'] + INTERCEPT + SLOPE * np.ceil(max(p + x['P'], x['V'] / DIM_FACTOR))

[costs, pounds] = RunCartonization(es_df, ul_bx)
costs.to_csv(PATH + 'costs.csv')
pounds.to_csv(PATH + 'pounds.csv')
[box_costs, tran_costs] = BreakCosts(costs, pounds, ul_bx)

[greedy_search, output] = GreedySearch(ul_bx, costs, tran_costs)

[parity_count, h_c, h_b, h_t] = BreakevenPoint(df, costs, t, greedy_search)

suite_sizes = [parity_count, int((parity_count + CURR_N_BOXES) / 2), CURR_N_BOXES, MAX_N_BOXES]

bao = RunOptimization(costs, suite_sizes)

utilizations = []
print('size, transp, box, total, volume')
print('baseline')
print(CURR_N_BOXES, round(h_c), round(h_b), round(h_t), round(t[['V_i', 'V_s']].max(axis = 1).sum() / 1728))
print('optimized')
ord_with_cost = list(set(t['o']).intersection(set(costs.index)).intersection(set(df[df['n'] != 'none']['o'])))
for size in suite_sizes:
    utilization = Utilization(costs.loc[ord_with_cost], pounds.loc[ord_with_cost], bao[size][2], ul_bx)
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume)


unique orders 3513
100 87.2742247581482


<ipython-input-28-cd16c0712a92>:107: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['tt'] = cost[['t']].sum(axis = 1)
<ipython-input-28-cd16c0712a92>:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
<ipython-input-28-cd16c0712a92>:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

200 159.5937888622284


<ipython-input-28-cd16c0712a92>:107: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['tt'] = cost[['t']].sum(axis = 1)
<ipython-input-28-cd16c0712a92>:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
<ipython-input-28-cd16c0712a92>:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

300 442.5227646827698


...

3100 2367.706815958023
3200 2383.1217617988586


<ipython-input-28-cd16c0712a92>:107: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['tt'] = cost[['t']].sum(axis = 1)
<ipython-input-28-cd16c0712a92>:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
<ipython-input-28-cd16c0712a92>:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

3300 2427.298847913742
3400 2446.3195538520813


<ipython-input-28-cd16c0712a92>:107: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['tt'] = cost[['t']].sum(axis = 1)
<ipython-input-28-cd16c0712a92>:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cost['pt'] = cost[['p']].apply(lambda x: json.dumps(sum([[round(e, 2) for e in json.loads(y)] for y in x if y is not None], [])), axis = 1)
<ipython-input-28-cd16c0712a92>:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

3500 2676.200761795044
2678.3149058818817
30 374.03105306625366
27 689.8173687458038
25 1070.16415476799
30 1441.3957738876343
size, transp, box, total, volume
baseline
25 655547 46278 701825 47132
optimized
30 686391 86089 772480 50046
27 689323 85395 774718 50842
25 690326 86292 776618 51147
30 686391 86089 772480 50046


In [4]:
costs.to_csv(PATH + 'costs.csv')
pounds.to_csv(PATH + 'pounds.csv')

utilizations[0].to_csv(PATH + 'utilization.csv')


In [44]:
ul_bx = pd.read_csv(PATH + 'arka_boxes.csv')

df = pd.read_csv(PATH + 'data.csv')
df_baseline = pd.read_csv(PATH + 'data_baseline.csv')
es_df = df[['o', 'l', 'w', 'h', 'q', 'p']].copy()

[SLOPE, INTERCEPT, t, baseline_ba] = InferTranCost(df_baseline)

In [46]:
t.sum()

o                KATY-54970KATY-54971KATY-54966KATY-54987SS-317...
Shipment Cost                                             711111.3
Box Cost                                                  46278.25
BW                                                        741451.0
V_i                                                            0.0
V_s                                                  89558938.2418
dtype: object

In [48]:
89558938.2418 / 1728

51828.08926030092

In [36]:
is_box = pd.concat([df_baseline[['n']], df_baseline[['L', 'W', 'H']].apply(lambda x: 
    'x'.join([str(c) for c in x]), axis = 1)], axis = 1).drop_duplicates().groupby('n').count()
act_box_data = df_baseline[['n', 'L', 'W', 'H']].drop_duplicates().merge(is_box[is_box[0] == 1].reset_index()[['n']], on = 'n', how = 'right')
match = BoxMatch(act_box_data, ul_bx, {'L_x' : 'L_y', 'W_x' : 'W_y', 'H_x' : 'H_y'}, 'n_x', ['c'], 2)
baseline_match = list(match.loc[3:]['n_y'])

In [38]:

utilization = Utilization(costs.loc[ord_with_cost], pounds.loc[ord_with_cost], baseline_match, ul_bx)
utilizations.append(utilization)
tot_cost = round(utilization['t'].sum())
box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume)

30 805456 122207 927663 71483


In [39]:
for i in range(len(utilizations)):
    utilizations[i].to_csv(PATH + 'utilization' + str(i) + '.csv')

In [40]:
greedy_search

,cumulative_cost,transport_cost,incremental_box,L,W,H,c
box_num,,,,,,,
1,3698720.52,2945232.0,S-4848,48.0,24.0,12.0,9.21
2,1222250.26,1043935.2,S-4423,14.0,12.0,6.0,1.03
3,966272.91,850672.8,S-4221,22.0,14.0,12.0,1.92
4,912362.76,811873.2,S-4121,12.0,8.0,6.0,0.68
5,886625.29,790612.8,S-4572,30.0,24.0,24.0,9.01
6,862625.98,764436.0,S-4905,24.0,14.0,6.0,2.49
7,847722.98,750993.9,S-4138,14.0,6.0,6.0,0.68
8,833339.15,738114.3,S-4219,24.0,18.0,12.0,2.70
9,822362.26,726318.9,S-4716,13.0,10.0,4.0,0.89


In [47]:
#parity_count = 17

suite_sizes2 = [18, 20, 22, 24]#[parity_count, int((parity_count + CURR_N_BOXES) / 2)]

bao2 = RunOptimization(costs, suite_sizes2)

for size in suite_sizes2:
    utilization = Utilization(costs.loc[ord_with_cost], pounds.loc[ord_with_cost], bao2[size][2], ul_bx)
    utilizations.append(utilization)
    tot_cost = round(utilization['t'].sum())
    box_cost = round((utilization['Total Utilization'] * utilization['c']).sum())
    box_volume = round((utilization['Total Utilization'] * utilization['L'] * utilization['W'] * utilization['H']).sum() / 1728)
    print(size, tot_cost - box_cost, box_cost, tot_cost, box_volume)
    
for i in range(len(utilizations)):
    utilizations[i].to_csv(PATH + 'utilization' + str(i) + '.csv')

18 353.0438401699066
20 729.7873492240906
22 1133.06014418602
24 1513.046399116516
18 696233 88710 784943 53270
20 693481 88563 782044 52532
22 692440 87375 779815 52018
24 691985 85689 777674 51108
